In [2]:
import os
import pandas as pd
import xarray as xr

# Dossiers où sont stockés les fichiers
data_dirs = {
    "Chlorophylle (mg/m³)": "./my_data_folder/chl_folder",
    "Température SST (°C)": "./my_data_folder/sst_folder",
    "Niveau de la mer (m)": "./my_data_folder/sea_level_folder"
}

# Initialiser un DataFrame vide
df_list = []

# Boucle sur chaque variable pour extraire les données
for var_name, directory in data_dirs.items():
    for file_name in os.listdir(directory):
        if file_name.endswith(".nc"):  # Vérifier que c'est un fichier NetCDF
            file_path = os.path.join(directory, file_name)
            ds = xr.open_dataset(file_path)
            
            # Extraction des variables : temps et valeur moyenne
            time_var = ds["time"].values  # Variable temps
            data_var = ds[list(ds.data_vars)[0]]  # Prendre la première variable

            # Si les données ont des dimensions spatiales, on fait la moyenne spatiale
            if "latitude" in data_var.dims and "longitude" in data_var.dims:
                data_mean = data_var.mean(dim=["latitude", "longitude"]).values
            else:
                data_mean = data_var.values  # Si pas de dimensions spatiales

            # Ajouter au DataFrame
            df_temp = pd.DataFrame({"Date": time_var, var_name: data_mean})
            df_list.append(df_temp)

# Fusionner les DataFrames
df_final = pd.concat(df_list, axis=0).sort_values("Date")
df_final = df_final.groupby("Date").mean().reset_index()

# Afficher le tableau final
print(df_final)

            Date  Chlorophylle (mg/m³)  Température SST (°C)  \
0     1982-01-15                   NaN             -0.228126   
1     1982-02-15                   NaN             -0.251763   
2     1982-03-15                   NaN             -0.639396   
3     1982-04-15                   NaN             -0.828476   
4     1982-05-15                   NaN             -1.347137   
...          ...                   ...                   ...   
11255 2023-10-15                   NaN              1.409275   
11256 2023-11-01              0.070774                   NaN   
11257 2023-11-15                   NaN              1.348947   
11258 2023-12-01              0.092599                   NaN   
11259 2023-12-15                   NaN              0.966553   

       Niveau de la mer (m)  
0                       NaN  
1                       NaN  
2                       NaN  
3                       NaN  
4                       NaN  
...                     ...  
11255                

In [3]:
first_last_dates = {}

# Boucle sur chaque variable dans le DataFrame
for column in df_final.columns[1:]:  # Ignorer la colonne "Date"
    # Exclure les NaN
    valid_data = df_final[df_final[column].notna()]
    
    # Obtenir la première et la dernière date où la valeur est non-NaN
    first_date = valid_data["Date"].iloc[0] if not valid_data.empty else None
    last_date = valid_data["Date"].iloc[-1] if not valid_data.empty else None
    
    # Stocker dans le dictionnaire
    first_last_dates[column] = {
        "First Date": first_date,
        "Last Date": last_date
    }

# Afficher les résultats
print("\nPremière et dernière date pour chaque variable :")
for var, dates in first_last_dates.items():
    print(f"{var} -> Première date: {dates['First Date']}, Dernière date: {dates['Last Date']}")


Première et dernière date pour chaque variable :
Chlorophylle (mg/m³) -> Première date: 1997-09-01 00:00:00, Dernière date: 2023-12-01 00:00:00
Température SST (°C) -> Première date: 1982-01-15 00:00:00, Dernière date: 2023-12-15 00:00:00
Niveau de la mer (m) -> Première date: 1993-01-01 00:00:00, Dernière date: 2023-06-07 00:00:00


In [4]:
# Spécifier le chemin du fichier où tu veux enregistrer les données
output_csv_path = "./my_data_folder/df_final.csv"

# Enregistrer le DataFrame dans un fichier CSV
df_final.to_csv(output_csv_path, index=False)

# Confirmer l'enregistrement
print(f"Le tableau a été enregistré dans {output_csv_path}")

Le tableau a été enregistré dans ./my_data_folder/df_final.csv


In [5]:
# Filtrer les lignes où il y a des valeurs non nulles pour les trois variables
df_filtered = df_final.dropna(subset=["Chlorophylle (mg/m³)", "Température SST (°C)", "Niveau de la mer (m)"])

# Afficher le tableau filtré
print(df_filtered)

# Enregistrer le DataFrame filtré dans un fichier (CSV, Excel ou Parquet)
output_csv_path_filtered = "./my_data_folder/df_final_filtered.csv"
df_filtered.to_csv(output_csv_path_filtered, index=False)

# Confirmer l'enregistrement
print(f"Le tableau filtré a été enregistré dans {output_csv_path_filtered}")

Empty DataFrame
Columns: [Date, Chlorophylle (mg/m³), Température SST (°C), Niveau de la mer (m)]
Index: []
Le tableau filtré a été enregistré dans ./my_data_folder/df_final_filtered.csv


In [7]:
import os
import pandas as pd
import xarray as xr

# Dossiers où sont stockés les fichiers
data_dirs = {
    "Chlorophylle (mg/m³)": "./my_data_folder/chl_folder",
    "Température SST (°C)": "./my_data_folder/sst_folder",
    "Niveau de la mer (m)": "./my_data_folder/sea_level_folder"
}

# Initialiser un DataFrame vide
df_list = []

# Boucle sur chaque variable pour extraire les données
for var_name, directory in data_dirs.items():
    for file_name in os.listdir(directory):
        if file_name.endswith(".nc"):  # Vérifier que c'est un fichier NetCDF
            file_path = os.path.join(directory, file_name)
            ds = xr.open_dataset(file_path)
            
            # Extraction de la variable "time"
            time_var = ds["time"].values  # Variable temps
            
            # Traitement spécifique pour les données du niveau de la mer
            if var_name == "Niveau de la mer (m)":
                # Récupérer la variable de niveau de la mer
                data_var = ds['MSL_filtered_GIA_TPA_corrected_adjusted']  # Variable de niveau de la mer

                # Calcul de la moyenne si des dimensions spatiales existent
                if "latitude" in data_var.dims and "longitude" in data_var.dims:
                    data_mean = data_var.mean(dim=["latitude", "longitude"]).values
                else:
                    data_mean = data_var.values  # Si pas de dimensions spatiales
            else:
                # Pour les autres variables, on prend la première variable
                data_var = ds[list(ds.data_vars)[0]]  # Prendre la première variable

                # Si les données ont des dimensions spatiales, on fait la moyenne spatiale
                if "latitude" in data_var.dims and "longitude" in data_var.dims:
                    data_mean = data_var.mean(dim=["latitude", "longitude"]).values
                else:
                    data_mean = data_var.values  # Si pas de dimensions spatiales

            # Ajouter au DataFrame
            df_temp = pd.DataFrame({"Date": time_var, var_name: data_mean})
            df_list.append(df_temp)

# Fusionner les DataFrames
df_final = pd.concat(df_list, axis=0).sort_values("Date")
df_final = df_final.groupby("Date").mean().reset_index()

# Afficher le tableau final
print(df_final)

# Sauvegarder le tableau en CSV (facultatif)
df_final.to_csv('./my_data_folder/combined_data.csv', index=False)

            Date  Chlorophylle (mg/m³)  Température SST (°C)  \
0     1982-01-15                   NaN                -0.228   
1     1982-02-15                   NaN                -0.252   
2     1982-03-15                   NaN                -0.639   
3     1982-04-15                   NaN                -0.828   
4     1982-05-15                   NaN                -1.347   
...          ...                   ...                   ...   
11255 2023-10-15                   NaN                 1.409   
11256 2023-11-01                 0.071                   NaN   
11257 2023-11-15                   NaN                 1.349   
11258 2023-12-01                 0.093                   NaN   
11259 2023-12-15                   NaN                 0.967   

       Niveau de la mer (m)  
0                       NaN  
1                       NaN  
2                       NaN  
3                       NaN  
4                       NaN  
...                     ...  
11255                

In [ ]:
########## Regroupement de toutes les données après les avoir mises au format csv précédemment

import os
import pandas as pd

# Définir les dossiers contenant les fichiers CSV
folders = [
    "./my_data_folder/sea_level_folder\sea_level_monthly.csv",   # Dossier des données journalières
    "./my_data_folder/sst_folder", # Dossier des données mensuelles
    "./my_data_folder/chl_folder"    # Autre dossier
]

dataframes = []

# Lire tous les fichiers CSV des trois dossiers
for folder in folders:
    if os.path.exists(folder):  # Vérifier si le dossier existe
        for file in os.listdir(folder):
            if file.endswith(".csv"):  # Vérifier si c'est un fichier CSV
                file_path = os.path.join(folder, file)
                df = pd.read_csv(file_path)

                # Convertir la colonne 'time' en format datetime
                df['time'] = pd.to_datetime(df['time'], errors='coerce')

                dataframes.append(df)

# Vérifier qu'il y a bien des fichiers à fusionner
if len(dataframes) < 2:
    print("Il faut au moins 2 fichiers CSV pour effectuer la fusion.")
else:
    # Fusionner toutes les bases sur la colonne 'time' en conservant toutes les valeurs
    merged_df = dataframes[0]
    for df in dataframes[1:]:
        merged_df = pd.merge(merged_df, df, on="time", how="outer")

    # Trier par ordre chronologique
    merged_df = merged_df.sort_values(by="time")

    # Enregistrer le fichier fusionné
    merged_csv_path = "./my_data_folder/merged_data.csv"
    merged_df.to_csv(merged_csv_path, index=False)

    print(f"Fichier fusionné enregistré dans {merged_csv_path}")
    print(merged_df.head())  # Afficher un aperçu des données fusionnées

Fichier fusionné enregistré dans ./my_data_folder/merged_data.csv
        time  MSL_filtered_GIA_TPA_corrected_adjusted  \
0 1982-01-15                                      NaN   
1 1982-02-15                                      NaN   
2 1982-03-15                                      NaN   
3 1982-04-15                                      NaN   
4 1982-05-15                                      NaN   

   trend_MSL_filtered_GIA_TPA_corrected_adjusted  sst_anomaly  \
0                                            NaN    -0.228126   
1                                            NaN    -0.251763   
2                                            NaN    -0.639396   
3                                            NaN    -0.828476   
4                                            NaN    -1.347137   

   sst_anomaly_filtered  chlor_a  
0                   NaN      NaN  
1                   NaN      NaN  
2                   NaN      NaN  
3                   NaN      NaN  
4                   NaN  

In [7]:
######## Réarrangement des colonnes
import pandas as pd

# Charger le fichier fusionné
merged_csv_path = "my_data_folder/merged_data.csv"
df = pd.read_csv(merged_csv_path)

# Vérifier les colonnes existantes
print("Colonnes disponibles :", df.columns.tolist())

# Spécifier le nouvel ordre des colonnes (adapter selon tes besoins)
new_order = ["time", "MSL_filtered_GIA_TPA_corrected_adjusted",'chlor_a', "sst_anomaly", "sst_anomaly_filtered",'trend_MSL_filtered_GIA_TPA_corrected_adjusted']  # Remplace par tes colonnes

# Réorganiser les colonnes
df = df[[col for col in new_order if col in df.columns] + 
        [col for col in df.columns if col not in new_order]]

# Sauvegarder le fichier avec le nouvel ordre
df.to_csv(merged_csv_path, index=False)

print(f"Fichier réorganisé enregistré dans {merged_csv_path}")
print(df.head())  # Afficher un aperçu du fichier

Colonnes disponibles : ['time', 'MSL_filtered_GIA_TPA_corrected_adjusted', 'trend_MSL_filtered_GIA_TPA_corrected_adjusted', 'sst_anomaly', 'sst_anomaly_filtered', 'chlor_a']
Fichier réorganisé enregistré dans my_data_folder/merged_data.csv
         time  MSL_filtered_GIA_TPA_corrected_adjusted  chlor_a  sst_anomaly  \
0  1982-01-15                                      NaN      NaN    -0.228126   
1  1982-02-15                                      NaN      NaN    -0.251763   
2  1982-03-15                                      NaN      NaN    -0.639396   
3  1982-04-15                                      NaN      NaN    -0.828476   
4  1982-05-15                                      NaN      NaN    -1.347137   

   sst_anomaly_filtered  trend_MSL_filtered_GIA_TPA_corrected_adjusted  
0                   NaN                                            NaN  
1                   NaN                                            NaN  
2                   NaN                                     

In [ ]:
import pandas as pd

####### On ne garde que les 3 premières colonnes

# Charger le fichier fusionné
merged_csv_path = "my_data_folder/merged_data.csv"
df = pd.read_csv(merged_csv_path)

# Sélectionner les 4 premières colonnes
df_filtered = df.iloc[:, :4]

# Sauvegarder le nouveau fichier CSV
filtered_csv_path = "./data_time_SL_chlor_SST.csv"
df_filtered.to_csv(filtered_csv_path, index=False)

print(f"Fichier avec 4 colonnes enregistré dans {filtered_csv_path}")
print(df_filtered.head())  # Afficher un aperçu des données

Fichier avec 4 colonnes enregistré dans ./data_time_SL_chlor_SST.csv
         time  MSL_filtered_GIA_TPA_corrected_adjusted  chlor_a  sst_anomaly
0  1982-01-15                                      NaN      NaN    -0.228126
1  1982-02-15                                      NaN      NaN    -0.251763
2  1982-03-15                                      NaN      NaN    -0.639396
3  1982-04-15                                      NaN      NaN    -0.828476
4  1982-05-15                                      NaN      NaN    -1.347137


In [16]:
######## Je mensualise les données seal level

import pandas as pd

# Charger le fichier CSV
csv_path = "my_data_folder/sea_level_folder/omi_climate_sl_medsea_area_averaged_anomalies_19930101_P20240228.csv"  # Remplace par ton chemin
df = pd.read_csv(csv_path)

# Convertir la colonne 'time' en format datetime
df['time'] = pd.to_datetime(df['time'], errors='coerce')

# Vérifier les données
print("Aperçu des données avant traitement :", df.head())

# Supprimer les lignes avec des dates invalides (NaT) après conversion
df = df.dropna(subset=['time'])

# Grouper par mois et calculer la moyenne des valeurs (en fonction de la colonne à moyenner)
df_monthly = df.groupby(df['time'].dt.to_period('M')).agg('mean').reset_index(drop=True)

print(df_monthly.head())

df_monthly.to_csv("my_data_folder/sea_level_folder/sea_level_monthly.csv", index=False)


Aperçu des données avant traitement :         time  MSL_filtered_GIA_TPA_corrected_adjusted  \
0 1993-01-01                                -2.440607   
1 1993-01-02                                -2.412209   
2 1993-01-03                                -2.383745   
3 1993-01-04                                -2.355198   
4 1993-01-05                                -2.326554   

   trend_MSL_filtered_GIA_TPA_corrected_adjusted  
0                                       2.000317  
1                                       2.000948  
2                                       2.001580  
3                                       2.002212  
4                                       2.002843  
                 time  MSL_filtered_GIA_TPA_corrected_adjusted  \
0 1993-01-16 00:00:00                                -1.991721   
1 1993-02-14 12:00:00                                -0.955554   
2 1993-03-16 00:00:00                                 0.316527   
3 1993-04-15 12:00:00                            

In [18]:
########## Regroupement de toutes les données après les avoir mises au format csv précédemment avec sea level monthly

import os
import pandas as pd

# Définir les dossiers contenant les fichiers CSV
folders = [
    "./my_data_folder/sea_level_folder/sea_level_monthly",   # Dossier des données journalières
    "./my_data_folder/sst_folder", # Dossier des données mensuelles
    "./my_data_folder/chl_folder"    # Autre dossier
]

dataframes = []

# Lire tous les fichiers CSV des trois dossiers
for folder in folders:
    if os.path.exists(folder):  # Vérifier si le dossier existe
        for file in os.listdir(folder):
            if file.endswith(".csv"):  # Vérifier si c'est un fichier CSV
                file_path = os.path.join(folder, file)
                df = pd.read_csv(file_path)

                # Convertir la colonne 'time' en format datetime
                df['time'] = pd.to_datetime(df['time'], errors='coerce')

                dataframes.append(df)

# Vérifier qu'il y a bien des fichiers à fusionner
if len(dataframes) < 2:
    print("Il faut au moins 2 fichiers CSV pour effectuer la fusion.")
else:
    # Fusionner toutes les bases sur la colonne 'time' en conservant toutes les valeurs
    merged_df = dataframes[0]
    for df in dataframes[1:]:
        merged_df = pd.merge(merged_df, df, on="time", how="outer")

    # Trier par ordre chronologique
    merged_df = merged_df.sort_values(by="time")

    # Enregistrer le fichier fusionné
    merged_csv_path = "./my_data_folder/merged_data_monthly.csv"
    merged_df.to_csv(merged_csv_path, index=False)

    print(f"Fichier fusionné enregistré dans {merged_csv_path}")
    print(merged_df.head())  # Afficher un aperçu des données fusionnées

######## Réarrangement des colonnes
import pandas as pd

# Charger le fichier fusionné
merged_csv_path = "my_data_folder/merged_data_monthly.csv"
df = pd.read_csv(merged_csv_path)

# Vérifier les colonnes existantes
print("Colonnes disponibles :", df.columns.tolist())

# Spécifier le nouvel ordre des colonnes (adapter selon tes besoins)
new_order = ["time", "MSL_filtered_GIA_TPA_corrected_adjusted",'chlor_a', "sst_anomaly", "sst_anomaly_filtered",'trend_MSL_filtered_GIA_TPA_corrected_adjusted']  # Remplace par tes colonnes

# Réorganiser les colonnes
df = df[[col for col in new_order if col in df.columns] + 
        [col for col in df.columns if col not in new_order]]

# Sauvegarder le fichier avec le nouvel ordre
df.to_csv(merged_csv_path, index=False)

print(f"Fichier réorganisé enregistré dans {merged_csv_path}")
print(df.head())  # Afficher un aperçu du fichier

import pandas as pd

####### On ne garde que les 3 premières colonnes

# Charger le fichier fusionné
merged_csv_path = "my_data_folder/merged_data_monthly.csv"
df = pd.read_csv(merged_csv_path)

# Sélectionner les 4 premières colonnes
df_filtered = df.iloc[:, :4]

# Sauvegarder le nouveau fichier CSV
filtered_csv_path = "./data_time_SL_chlor_SST_monthly.csv"
df_filtered.to_csv(filtered_csv_path, index=False)

print(f"Fichier avec 4 colonnes enregistré dans {filtered_csv_path}")
print(df_filtered.head())  # Afficher un aperçu des données







Fichier fusionné enregistré dans ./my_data_folder/merged_data_monthly.csv
        time  MSL_filtered_GIA_TPA_corrected_adjusted  \
0 1982-01-15                                      NaN   
1 1982-02-15                                      NaN   
2 1982-03-15                                      NaN   
3 1982-04-15                                      NaN   
4 1982-05-15                                      NaN   

   trend_MSL_filtered_GIA_TPA_corrected_adjusted  sst_anomaly  \
0                                            NaN    -0.228126   
1                                            NaN    -0.251763   
2                                            NaN    -0.639396   
3                                            NaN    -0.828476   
4                                            NaN    -1.347137   

   sst_anomaly_filtered  chlor_a  
0                   NaN      NaN  
1                   NaN      NaN  
2                   NaN      NaN  
3                   NaN      NaN  
4                

In [21]:
import pandas as pd
# Charger le fichier CSV dans un DataFrame

df = pd.read_csv("my_data_folder/merged_data_monthly.csv")

# Convertir la colonne 'time' en format datetime
df['time'] = pd.to_datetime(df['time'], errors='coerce')

# Extraire l'année et le mois pour chaque ligne (nous utilisons .dt.to_period('M') pour avoir la période au format 'YYYY-MM')
df['year_month'] = df['time'].dt.to_period('M')

# Grouper par 'year_month' et prendre la première valeur disponible pour chaque mois
df_monthly = df.groupby('year_month').first().reset_index()

# Aperçu des données après sélection d'une valeur par mois
print(df_monthly.head())

# Si vous souhaitez sauvegarder le DataFrame sous forme de fichier CSV :
df_monthly.to_csv("merged_data_grouped.csv", index=False)

  year_month       time  MSL_filtered_GIA_TPA_corrected_adjusted  chlor_a  \
0    1982-01 1982-01-15                                      NaN      NaN   
1    1982-02 1982-02-15                                      NaN      NaN   
2    1982-03 1982-03-15                                      NaN      NaN   
3    1982-04 1982-04-15                                      NaN      NaN   
4    1982-05 1982-05-15                                      NaN      NaN   

   sst_anomaly  sst_anomaly_filtered  \
0    -0.228126                   NaN   
1    -0.251763                   NaN   
2    -0.639396                   NaN   
3    -0.828476                   NaN   
4    -1.347137                   NaN   

   trend_MSL_filtered_GIA_TPA_corrected_adjusted  
0                                            NaN  
1                                            NaN  
2                                            NaN  
3                                            NaN  
4                                        

In [5]:
######## Convertir les données de ice mass pour avoir la bonne échelle de temps
import pandas as pd
import datetime

def decimal_year_to_ym(decimal_year):
    year = int(decimal_year)
    remainder = decimal_year - year
    start_of_year = datetime.datetime(year, 1, 1)
    days_in_year = (datetime.datetime(year + 1, 1, 1) - start_of_year).days
    date = start_of_year + datetime.timedelta(days=remainder * days_in_year)
    return date.strftime('%Y-%m') 

df = pd.read_csv("my_data_folder/Icemass_folder/merged_mass_data.csv")
df['year_month'] = df['time'].apply(decimal_year_to_ym)
cols = ['year_month'] + [col for col in df.columns if col != 'year_month']
df = df[cols]

print(df)

df.to_csv("my_data_folder/Icemass_folder/merged_mass_data_ym.csv", index=False)

    year_month     time  greenland_mass_antarctica  uncertainty_antarctica  \
0      2002-04  2002.29                       0.00                  134.71   
1      2002-05  2002.35                      63.95                   70.85   
2      2002-08  2002.62                    -215.57                   53.55   
3      2002-09  2002.71                    -235.55                   65.24   
4      2002-10  2002.79                    -201.92                   39.23   
..         ...      ...                        ...                     ...   
236    2024-09  2024.71                   -5595.38                   36.43   
237    2024-10  2024.79                   -5614.00                   47.66   
238    2024-11  2024.87                   -5607.44                   59.05   
239    2024-12  2024.96                   -5610.46                   70.87   
240    2025-01  2025.04                   -5608.47                   84.32   

     greenland_mass_greenland  uncertainty_greenland  
0       

In [13]:
######## Convertir la base de donnée csv pour avoir une seule valeur par mois##############
import pandas as pd
import pandas as pd

# Charger le fichier CSV
df = pd.read_csv("my_data_folder/CO2_folder/co2_data.csv")

# Renommer les colonnes
df.columns = ["year", "month", "day", "value"]

# Supprimer les lignes contenant du texte (en supposant que les valeurs numériques sont attendues)
df = df[pd.to_numeric(df["year"], errors="coerce").notna()]


df = df.reset_index(drop=True)

print(df.head())

df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["month"] = pd.to_numeric(df["month"], errors="coerce")
df["day"] = pd.to_numeric(df["day"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")

df = df.dropna()# Supprime Nan

df['year_month'] = df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2)

df_grouped = df.groupby('year_month').agg({'value': 'mean'}).reset_index()

print(df_grouped)
df_grouped.to_csv("my_data_folder/CO2_folder/co2_data_reformated.csv", index=False)


   year month day   value
0  2006    10  12  383.38
1  2006    10  19  379.58
2  2006    10  24  381.02
3  2006    11   2  385.48
4  2006    11   9  382.09
    year_month       value
0      2006-10  381.326667
1      2006-11  384.568000
2      2006-12  385.715000
3      2007-01  386.762500
4      2007-02  386.982500
..         ...         ...
212    2024-07  420.610000
213    2024-08  422.105000
214    2024-09  419.763333
215    2024-10  423.876667
216    2024-11  425.070000

[217 rows x 2 columns]


In [14]:
######## Regrouper avec les données de CO2 et de ice mass ##########

import pandas as pd

# Charger les trois fichiers CSV dans des DataFrames
df1 = pd.read_csv('my_data_folder/bdd_utilisable/data_monthly_time_sl_chlor_sst_sstfiltered_slfiltered.csv')
df2 = pd.read_csv('my_data_folder/CO2_folder/co2_data_reformated.csv')
df3 = pd.read_csv('my_data_folder/Icemass_folder/merged_mass_data_ym.csv')


# Fusionner les DataFrames sur la colonne 'year_month'
merged_df = pd.merge(df1, df2, on='year_month', how='outer')
merged_df = pd.merge(merged_df, df3, on='year_month', how='outer')

# Enregistrer le DataFrame fusionné dans un fichier CSV
merged_df.to_csv('my_data_folder/bdd_utilisable/merged_data.csv', index=False)

# Afficher un aperçu du DataFrame fusionné
print(merged_df.head())

  year_month               time_x  MSL_filtered_GIA_TPA_corrected_adjusted  \
0    1982-01  1982-01-15 00:00:00                                      NaN   
1    1982-02  1982-02-15 00:00:00                                      NaN   
2    1982-03  1982-03-15 00:00:00                                      NaN   
3    1982-04  1982-04-15 00:00:00                                      NaN   
4    1982-05  1982-05-15 00:00:00                                      NaN   

   chlor_a  sst_anomaly  sst_anomaly_filtered  \
0      NaN    -0.228126                   NaN   
1      NaN    -0.251763                   NaN   
2      NaN    -0.639396                   NaN   
3      NaN    -0.828476                   NaN   
4      NaN    -1.347137                   NaN   

   trend_MSL_filtered_GIA_TPA_corrected_adjusted  value  time_y  \
0                                            NaN    NaN     NaN   
1                                            NaN    NaN     NaN   
2                                  

In [15]:
import pandas as pd

# Charger votre dataframe à partir du fichier CSV
df = pd.read_csv('my_data_folder/bdd_utilisable/merged_data.csv')

# Supprimer les colonnes time_x et time_y
df.drop(['time_x', 'time_y', 'trend_MSL_filtered_GIA_TPA_corrected_adjusted',"uncertainty_antarctica","greenland_mass_greenland","uncertainty_greenland","sst_anomaly"], axis=1, inplace=True)

df.to_csv('my_data_folder/bdd_utilisable/clean_merged_data.csv', index=False)


print(df.head())

  year_month  MSL_filtered_GIA_TPA_corrected_adjusted  chlor_a  \
0    1982-01                                      NaN      NaN   
1    1982-02                                      NaN      NaN   
2    1982-03                                      NaN      NaN   
3    1982-04                                      NaN      NaN   
4    1982-05                                      NaN      NaN   

   sst_anomaly_filtered  value  greenland_mass_antarctica  
0                   NaN    NaN                        NaN  
1                   NaN    NaN                        NaN  
2                   NaN    NaN                        NaN  
3                   NaN    NaN                        NaN  
4                   NaN    NaN                        NaN  


In [16]:
######### Ajout de la salinité dans la bdd finale##########

import pandas as pd

# Charger les deux DataFrames
df1 = pd.read_csv('my_data_folder/bdd_utilisable/clean_merged_data.csv')  # Première base de données (year_month)
df2 = pd.read_csv("my_data_folder/salinity_folder/BDD_salinite.csv")  # Deuxième base de données (time)

df1['year_month'] = pd.to_datetime(df1['year_month'], format='%Y-%m')
df2['time'] = pd.to_datetime(df2['time'], format='%Y-%m')

merged_df = pd.merge(df1, df2, left_on='year_month', right_on='time', how='outer')

# Afficher les premières lignes du DataFrame fusionné
print(merged_df.head())

# Sauvegarder le DataFrame fusionné dans un nouveau fichier CSV
merged_df.to_csv('my_data_folder/bdd_utilisable/data_merged_CO2_SL_SST_Ice_Mass_Chlor.csv', index=False)

  year_month  MSL_filtered_GIA_TPA_corrected_adjusted  chlor_a  \
0 1982-01-01                                      NaN      NaN   
1 1982-02-01                                      NaN      NaN   
2 1982-03-01                                      NaN      NaN   
3 1982-04-01                                      NaN      NaN   
4 1982-05-01                                      NaN      NaN   

   sst_anomaly_filtered  value  greenland_mass_antarctica time  \
0                   NaN    NaN                        NaN  NaT   
1                   NaN    NaN                        NaN  NaT   
2                   NaN    NaN                        NaN  NaT   
3                   NaN    NaN                        NaN  NaT   
4                   NaN    NaN                        NaN  NaT   

   moyenne_variable  
0               NaN  
1               NaN  
2               NaN  
3               NaN  
4               NaN  


In [3]:
import pandas as pd

# Charger le fichier CSV
df = pd.read_csv("my_data_folder/bdd_utilisable/data_merged_CO2_SL_SST_Ice_Mass_Chlor.csv")

# Renommer les colonnes
df = df.rename(columns={"chlor_a": "chlorophylle", "moyenne_variable": "salinité", "MSL_filtered_GIA_TPA_corrected_adjusted":"sea level corrected adjusted", "value" : "CO2" })
df = df.drop(columns=["time"])

# Sauvegarder les modifications (optionnel)
df.to_csv("my_data_folder/bdd_utilisable/bdd_rename.csv", index=False)

# Vérifier le résultat
print(df.head())

   year_month  sea level corrected adjusted  chlorophylle  \
0  1982-01-01                           NaN           NaN   
1  1982-02-01                           NaN           NaN   
2  1982-03-01                           NaN           NaN   
3  1982-04-01                           NaN           NaN   
4  1982-05-01                           NaN           NaN   

   sst_anomaly_filtered  CO2  greenland_mass_antarctica  salinité  
0                   NaN  NaN                        NaN       NaN  
1                   NaN  NaN                        NaN       NaN  
2                   NaN  NaN                        NaN       NaN  
3                   NaN  NaN                        NaN       NaN  
4                   NaN  NaN                        NaN       NaN  
